<a href="https://colab.research.google.com/github/geopayme/AstroPhysics/blob/main/mineral_mapping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# mineral_mapping.py
import numpy as np
import pandas as pd
from PIL import Image
from skimage.color import rgb2lab
from sklearn.cluster import KMeans
from matplotlib.image import imsave

# === 1) Load the Perseverance image ===
# Adjust this path if needed to point at your saved Mastcam-Z PNG
src_path = "Mars_Perseverance_SIF_1476_0797970484_351EBY_N0720000SRLC00636_0000LMJ.png"
img = Image.open(src_path).convert("RGB")
arr = np.array(img)

# === 2) Convert to LAB and cluster ===
lab = rgb2lab(arr)
h, w, _ = lab.shape
lab_flat = lab.reshape((-1, 3))

km = KMeans(n_clusters=4, random_state=42).fit(lab_flat)
labels = km.labels_.reshape(h, w)

# === 3) Define mineral labels & colors ===
label_names = {
    0: "Olivine/Basaltic Sand",
    1: "Dust-Covered Regolith",
    2: "Sulfate-Rich Zone",
    3: "Clay-Bearing Outcrop"
}
colors = {
    0: (139, 69, 19),
    1: (210, 180, 140),
    2: (100, 149, 237),
    3: (178, 34, 34)
}

# === 4) Build annotated image ===
anno = np.zeros((h, w, 3), dtype=np.uint8)
for k, col in colors.items():
    anno[labels==k] = col

# Save the PNG
out_png = "Mineral_Map_Phase2_Annotated.png"
imsave(out_png, anno)
print(f"Annotated map saved to {out_png}")

# === 5) Build & save CSV ===
coords = [(y, x) for y in range(h) for x in range(w)]
classes = [label_names[labels[y, x]] for y, x in coords]
df = pd.DataFrame(coords, columns=["y","x"])
df["mineral_class"] = classes

out_csv = "Mars_Mineral_Classification_Phase2.csv"
df.to_csv(out_csv, index=False)
print(f"Pixel classification CSV saved to {out_csv}")


In [4]:
import numpy as np
from skimage.io import imread
from scipy.ndimage import generic_filter

# 1. Load the RGBA map and drop alpha
cm_rgba = imread("Mineral_Map_Phase2_Annotated.png")
cm = cm_rgba[..., :3]            # keep only RGB
h, w, _ = cm.shape

# 2. Remap colors to class IDs
lut = {
    (139,  69,  19): 0,   # Olivine/Basalt
    (210, 180, 140): 1,   # Dust-Regolith
    (100, 149, 237): 2,   # Sulfate
    (178,  34,  34): 3    # Clay
}
id_map = np.zeros((h, w), dtype=int)
for rgb, cid in lut.items():
    # broadcast comparison against the first three channels only
    mask = np.all(cm == rgb, axis=-1)
    id_map[mask] = cid

# 3. Compute neighbor‐agree score
def neighbor_agree(patch):
    center = patch[4]
    # exclude negative padding
    valid = patch >= 0
    return np.mean(patch[valid] == center)

agree_map = generic_filter(
    id_map,
    neighbor_agree,
    size=(3,3),
    mode='constant',
    cval=-1
)

# 4. Average per class
for cid in np.unique(id_map):
    vals = agree_map[id_map == cid]
    vals = vals[vals >= 0]
    score = np.mean(vals) if len(vals) else np.nan
    print(f"Class {cid} neighbor‐agree: {score:.3f}")


Class 0 neighbor‐agree: 0.620
Class 1 neighbor‐agree: 0.778
Class 2 neighbor‐agree: 0.530
Class 3 neighbor‐agree: 0.647


In [5]:
import numpy as np
from skimage.morphology import opening, closing, square
from scipy.ndimage import generic_filter

# 1. Load and strip alpha as before
cm_rgba = imread("Mineral_Map_Phase2_Annotated.png")
cm = cm_rgba[..., :3]
h, w, _ = cm.shape

# 2. Remap colors → class ID
lut = {
    (139,  69,  19): 0,
    (210, 180, 140): 1,
    (100, 149, 237): 2,
    (178,  34,  34): 3
}
id_map = np.zeros((h, w), dtype=int)
for rgb, cid in lut.items():
    id_map[np.all(cm == rgb, axis=-1)] = cid

# 3. Majority-filter each pixel’s neighborhood:
def majority(patch):
    vals, counts = np.unique(patch, return_counts=True)
    return vals[np.argmax(counts)]
clean = generic_filter(id_map, majority, size=3, mode='nearest')

# 4. Optionally smooth jagged edges with small opening+closing
clean = opening(clean, square(3))
clean = closing(clean, square(3))

# 5. Recompute neighbor-agree
def neighbor_agree(patch):
    center = patch[4]
    return np.mean(patch == center)

agree = generic_filter(clean, neighbor_agree, size=3, mode='constant', cval=-1)

for cid in range(4):
    vals = agree[clean==cid]
    vals = vals[vals>=0]
    print(f"Class {cid} cleaned neighbor-agree: {np.mean(vals):.3f}")


<ipython-input-5-ff89630185b2>:28: FutureWarning: `square` is deprecated since version 0.25 and will be removed in version 0.27. Use `skimage.morphology.footprint_rectangle` instead.
  clean = opening(clean, square(3))
<ipython-input-5-ff89630185b2>:29: FutureWarning: `square` is deprecated since version 0.25 and will be removed in version 0.27. Use `skimage.morphology.footprint_rectangle` instead.
  clean = closing(clean, square(3))


Class 0 cleaned neighbor-agree: 0.813
Class 1 cleaned neighbor-agree: 0.859
Class 2 cleaned neighbor-agree: 0.634
Class 3 cleaned neighbor-agree: 0.804
